# Minor Project

### Spam Detection using Classification Algorithm

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [2]:
df = pd.read_csv("enron_spam_data.csv")

In [3]:
df.head()

,Unnamed: 0,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14


In [4]:
df.shape

(33716, 5)

In [5]:
df.isnull().sum()

Unnamed: 0     0
Subject        0
Message       52
Spam/Ham       0
Date           0
dtype: int64

In [6]:
df.dropna()

,Unnamed: 0,Subject,Message,Spam/Ham,Date
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
5,5,mcmullen gas for 11 / 99,"jackie ,\nsince the inlet to 3 river plant is ...",ham,1999-12-14
...,...,...,...,...,...
33711,33711,fw : re ivanhoe e . s . d,"fyi , kim .\n- - - - - original message - - - ...",spam,2005-07-29
33712,33712,fw : re ivanhoe e . s . d,"fyi , kim .\n- - - - - original message - - - ...",spam,2005-07-29
33713,33713,fw : re ivanhoe e . s . d,"fyi , kim .\n- - - - - original message - - - ...",spam,2005-07-30
33714,33714,fw : re ivanhoe e . s . d,"fyi , kim .\n- - - - - original message - - - ...",spam,2005-07-30


In [7]:
df=df.drop(df.columns[0:1],axis=1)

In [8]:
print(f"Before dedup: {len(df)}")
df = df.drop_duplicates()
print(f"After dedup: {len(df)}")

Before dedup: 33716
After dedup: 18278


In [9]:
df=df.drop("Date",axis=1)

In [10]:
df.head()

,Subject,Message,Spam/Ham
0,christmas tree farm pictures,NaN,ham
1,"vastar resources , inc .","gary , production from the high island larger ...",ham
2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham
3,re : issue,fyi - see note below - already done .\nstella\...,ham
4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham


In [11]:
X=(
    df["Subject"].fillna("")+ " " +
    df["Message"].fillna("")
)
y=df["Spam/Ham"]

In [12]:
X

0                            christmas tree farm pictures 
1        vastar resources , inc . gary , production fro...
2        calpine daily gas nomination - calpine daily g...
3        re : issue fyi - see note below - already done...
4        meter 7268 nov allocation fyi .\n- - - - - - -...
                               ...                        
33663    fw : re ivanhoe e . s . d fyi , kim .\n- - - -...
33684    fw : re ivanhoe e . s . d fyi , kim .\n- - - -...
33694    fw : re ivanhoe e . s . d fyi , kim .\n- - - -...
33713    fw : re ivanhoe e . s . d fyi , kim .\n- - - -...
33715    fw : re ivanhoe e . s . d fyi , kim .\n- - - -...
Length: 18278, dtype: object

In [13]:
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

c:\Users\ishan\anaconda3\Lib\site-packages\sklearn\utils\_param_validation.py:14: UserWarning: A NumPy version >=1.23.5 and <2.5.0 is required for this version of SciPy (detected version 2.5.0)
  from scipy.sparse import csr_matrix, issparse


In [14]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.30, random_state=42)

In [15]:
X_train.shape

(12794,)

In [16]:
le = LabelEncoder()
le.fit(y_train)

LabelEncoder()

In [17]:
y_train_encoded=le.transform(y_train)
y_test_encoded=le.transform(y_test)

In [18]:
print(y_train[:5],y_train_encoded[:5])

11255    ham
3649     ham
2519     ham
23315    ham
5967     ham
Name: Spam/Ham, dtype: object [0 0 0 0 0]


In [19]:
vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1,2),
    stop_words='english',
    min_df=2,
    max_df=0.9,
    sublinear_tf=True
)

X_train_tfidf=vectorizer.fit_transform(X_train)
X_test_tfidf=vectorizer.transform(X_test)

In [20]:
print(len(vectorizer.vocabulary_))

5000


In [21]:
nb=MultinomialNB()
nb.fit(X_train_tfidf,y_train_encoded)
nb_predict=nb.predict(X_test_tfidf)

In [22]:
lr=LogisticRegression()
lr.fit(X_train_tfidf,y_train_encoded)
lr_predict=lr.predict(X_test_tfidf)

In [23]:
for name,preds in [("Naive_Bayes",nb_predict),("Logistic_Regression", lr_predict)]:
    print(f"\n{name}")
    print(classification_report(y_test_encoded,preds))
    print("ROC_AUC: ",roc_auc_score(y_test_encoded,lr.predict_proba(X_test_tfidf)[:,1] if name=="Logistic_Regression" else nb.predict_proba(X_test_tfidf)[:,1]))


Naive_Bayes
              precision    recall  f1-score   support

           0       1.00      0.98      0.99      4774
           1       0.89      1.00      0.94       710

    accuracy                           0.98      5484
   macro avg       0.94      0.99      0.97      5484
weighted avg       0.99      0.98      0.98      5484

ROC_AUC:  0.9995871121155083

Logistic_Regression
              precision    recall  f1-score   support

           0       1.00      1.00      1.00      4774
           1       0.99      1.00      1.00       710

    accuracy                           1.00      5484
   macro avg       1.00      1.00      1.00      5484
weighted avg       1.00      1.00      1.00      5484

ROC_AUC:  0.9998018905220177
